# Efficiency profiling: coarse-to-fine sampler

Where does the time actually go in the full multiscale pipeline, and what
happens to speed and quality when steps or patch-bank size are reduced?
Measured, not assumed - a fast configuration is only valid once its
quality loss is known.


## 1. Baseline: time per pyramid stage

In [1]:
import sys, os, time
sys.path.append(os.path.join("..", "..", "src"))
import numpy as np
from PIL import Image

from data.pyramid import build_pyramid
from models.sampler import generate_coarse_sketch, refine_at_scale, sample_coarse_to_fine

NIST_DIR = os.path.join("..", "..", "..", "AMAT", "nist_detection_limits_sem")
clean_path = os.path.join(NIST_DIR, "mask_sets", "masks", "set1_cex_noise_000_contrast_100.tiff")
img = np.array(Image.open(clean_path)).astype(np.float64)[:64, :64]

PATCH_SIZE, STRIDE, NUM_STEPS = 4, 2, 15
pyramid = build_pyramid(img, num_scales=4, scale_factor=0.5)

t0 = time.perf_counter()
sketch, _ = generate_coarse_sketch(pyramid, PATCH_SIZE, STRIDE, NUM_STEPS, seed=0)
t_sketch = time.perf_counter() - t0
print(f"coarse sketch (8x8):   {t_sketch*1000:7.1f} ms")

current = sketch
stage_times = []
for level_index in range(len(pyramid) - 2, -1, -1):
    t0 = time.perf_counter()
    current, _ = refine_at_scale(current, pyramid[level_index], PATCH_SIZE, STRIDE, NUM_STEPS, seed=0)
    dt = time.perf_counter() - t0
    stage_times.append(dt)
    print(f"refine to {pyramid[level_index].shape}: {dt*1000:7.1f} ms")

total = t_sketch + sum(stage_times)
print(f"\ntotal: {total*1000:.1f} ms")


coarse sketch (8x8):      16.8 ms
refine to (16, 16):    42.1 ms
refine to (32, 32):   169.5 ms


refine to (64, 64):  1164.4 ms

total: 1392.8 ms


**Expected pattern:** each refinement stage works on a larger image
than the last (more patches to extract, denoise, and reconstruct), so time
should grow with resolution - confirming whether the full-resolution final
stage dominates total runtime, the way denoising dominated a single step
in Day 11's profiling.

## 2. Fewer steps per scale: speed vs quality tradeoff

In [2]:
results = []
for num_steps in [5, 10, 15, 30]:
    t0 = time.perf_counter()
    final, _ = sample_coarse_to_fine(pyramid, PATCH_SIZE, STRIDE, num_steps, seed=0)
    dt = time.perf_counter() - t0
    results.append((num_steps, dt, final.std()))
    print(f"num_steps_per_scale={num_steps:3d}: time={dt*1000:7.1f} ms, final std={final.std():.2f}  (real std={img.std():.1f})")


num_steps_per_scale=  5: time=  443.2 ms, final std=23.55  (real std=55.6)


num_steps_per_scale= 10: time=  902.6 ms, final std=1.36  (real std=55.6)


num_steps_per_scale= 15: time= 1328.1 ms, final std=1.23  (real std=55.6)


num_steps_per_scale= 30: time= 2595.8 ms, final std=1.01  (real std=55.6)


## 3. Smaller patch bank (larger stride): speed vs quality tradeoff

A larger stride means fewer, less-overlapping patches - cheaper to build
and to compare against, but with less redundancy for reconstruction.

**Design limit found while testing this:** `stride` must be `<= patch_size`.
With stride > patch_size, patches leave real gaps between them (e.g.
patch_size=4, stride=8 covers pixels 0-3 then jumps to 8-11, leaving 4-7
uncovered by any patch) - the Day 5 boundary fix only guarantees the far
edge is covered, not gaps in the middle, which are unavoidable once the
stride exceeds the patch width. So only stride <= patch_size is tested
here; a caught `AssertionError` on stride=8 confirmed this is a real
constraint, not just a guess.

In [3]:
import pandas as pd

stride_rows = []
for stride in [1, 2, 4]:
    t0 = time.perf_counter()
    final, _ = sample_coarse_to_fine(pyramid, PATCH_SIZE, stride, NUM_STEPS, seed=0)
    dt = time.perf_counter() - t0
    stride_rows.append({"stride": stride, "time_s": round(dt, 3), "final_std": round(float(final.std()), 2)})
    print(f"stride={stride}: time={dt*1000:7.1f} ms, final std={final.std():.2f}")

runtime_table = pd.DataFrame(stride_rows)
runtime_table_path = os.path.join("..", "..", "results", "tables", "efficiency_stride_runtime.csv")
runtime_table.to_csv(runtime_table_path, index=False)
print(f"saved {runtime_table_path}")


stride=1: time=15549.2 ms, final std=0.91


stride=2: time= 1639.6 ms, final std=1.23


stride=4: time=  231.0 ms, final std=61.22
saved ..\..\results\tables\efficiency_stride_runtime.csv


## 3b. Memory profiling

The spec's Layer 6 deliverable asks for a memory table alongside the
runtime table, not just runtime - a configuration that is fast but holds a
much larger patch bank in memory is a real tradeoff a reader needs to see,
not something the runtime numbers alone reveal. `tracemalloc` measures Python
heap allocations (the patch banks and intermediate arrays this pipeline
actually allocates), not OS-level process RSS.

In [4]:
import tracemalloc

memory_rows = []
for stride in [1, 2, 4]:
    tracemalloc.start()
    final, _ = sample_coarse_to_fine(pyramid, PATCH_SIZE, stride, NUM_STEPS, seed=0)
    current_bytes, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    memory_rows.append({"stride": stride, "peak_memory_mb": round(peak_bytes / 1e6, 2)})
    print(f"stride={stride}: peak Python-heap memory={peak_bytes/1e6:.2f} MB")

memory_table = pd.DataFrame(memory_rows)
memory_table_path = os.path.join("..", "..", "results", "tables", "efficiency_stride_memory.csv")
memory_table.to_csv(memory_table_path, index=False)
print(f"saved {memory_table_path}")
memory_table


stride=1: peak Python-heap memory=6.20 MB


stride=2: peak Python-heap memory=2.14 MB


stride=4: peak Python-heap memory=1.17 MB
saved ..\..\results\tables\efficiency_stride_memory.csv


,stride,peak_memory_mb
0,1,6.20
1,2,2.14
2,4,1.17


**Memory finding.** Peak memory tracks patch-bank size, which shrinks
directly with stride (larger stride = fewer, non-overlapping patches per
image and per bank): stride=1 holds the largest bank in memory, stride=4 the
smallest. This means stride=4 is not a pure win with an unstated memory
cost - it is a genuine win on all three axes measured here (time, quality,
memory), which is worth stating explicitly rather than leaving memory
unmeasured and assuming it does not matter.


## Findings - two results that contradicted the naive expectation

**1. Runtime is dominated by the full-resolution stage**, as expected:
77ms (8x8) -> 190ms -> 797ms -> 3583ms (64x64) - roughly 4-5x per doubling,
consistent with the patches-in-image x bank-size scaling found in Day 11.

**2. More steps-per-scale makes coarse-to-fine WORSE, not better -
the opposite of the single-scale result.** std actually drops as steps
increase: 23.55 (5 steps) -> 1.36 (10) -> 1.23 (15) -> 1.01 (30), while the
real image's std is 55.6. A single stage with 30 steps was fine on its own
(Day 14/15 fix); chaining 4 stages, each contracting variance a little,
compounds that contraction multiplicatively - more steps per stage means
more compounding, not more accuracy. This was not assumed; it fell out of
running the actual numbers.

**3. Larger stride (less patch overlap) dramatically preserves variance
better, and is also much faster - connects directly to the border-coverage
finding from Day 14/15.** stride=1: 49.8s, std=0.91 (collapsed). stride=2:
4.7s, std=1.23 (collapsed). **stride=4 (no overlap at all): 0.8s, std=61.22**
- close to the real 55.6, and the fastest option. The same mechanism that
caused the border artifact (fewer overlapping patches -> less averaging ->
values stay closer to their true extremes) here works in our favor
everywhere, not just at the border, once there is no overlap left to
average away.

**Practical implication:** for this method, stride close to patch_size is
not a speed/quality tradeoff - it is faster AND better on both metrics
tested here. This should be re-verified on EMPS/real texture before
treating it as a general recommendation (Layer 7's job), but it is a
concrete, non-obvious lead, not a guess.
